# Render mean sparse and dense landmarks on the atlas mesh

Loads the atlas (mean) mesh together with its mean sparse landmarks and mean dense
correspondences, and produces:

* an interactive 3D view in the notebook (PyVista + trame, rendered server-side so it
  works without browser WebGL)
* an interactive Plotly scene written to `.html`, for viewing in a browser
* static screenshots from fixed camera angles (`.png`), rendered off-screen with PyVista

Outputs go to `<parent_dir>/landmark_renders/`.

In [ ]:
import os
from pathlib import Path
import numpy as np, plotly.graph_objects as go, pyvista as pv
from NSM.plotting import load_mrk_json
import plotly.io as pio
pio.renderers.default = "png"          
pio.kaleido.scope.default_scale = 2    
pv.set_jupyter_backend("server") 

ATLAS_DIR = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/"
                 "final_dataset_aug26/atlas/2026_07-15_13_06_22/atlas")
os.chdir(Path.cwd().parent)
OUT_DIR = Path("landmark_renders"); OUT_DIR.mkdir(exist_ok=True)

mesh = pv.read(str(ATLAS_DIR / "atlas_model.ply")).triangulate()
sparse, _ = load_mrk_json(ATLAS_DIR / "atlas_sparse_landmarks.mrk.json")
dense,  _ = load_mrk_json(ATLAS_DIR / "atlas_dense_correspondences.mrk.json")
print(mesh.n_points, "verts |", sparse.shape, "sparse |", dense.shape, "dense")

### Sparse landmarks

In [ ]:
# Sparse landmarks - html interactive

v, f = mesh.points, mesh.faces.reshape(-1, 4)[:, 1:]
fig = go.Figure([
    go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2],
              color="lightgray", opacity=0.45, hoverinfo="skip", showlegend=False),
    go.Scatter3d(x=sparse[:,0], y=sparse[:,1], z=sparse[:,2], mode="markers",
                 marker=dict(size=6, color="#D95319"), name=f"sparse (n={len(sparse):,})")])
fig.update_layout(title="Mean sparse landmarks on atlas mesh", width=900, height=700,
                  scene=dict(aspectmode="data"), margin=dict(l=0, r=0, b=0, t=40))
fig.write_html(str(OUT_DIR / "sparse_mean_landmarks.html"), include_plotlyjs="cdn")

In [ ]:
# In notebook render

p = pv.Plotter(window_size=(900, 700))
p.add_mesh(mesh, color="lightgray", opacity=0.45, smooth_shading=True)
p.add_points(sparse, color="#D95319", point_size=14,
             render_points_as_spheres=True, label="sparse")
p.add_legend()
p.show()

In [ ]:
# Screenshot -> PNG   (pv.start_xvfb() first if headless)
p = pv.Plotter(off_screen=True, window_size=(1600, 1200))
p.add_mesh(mesh, color="lightgray", opacity=0.45, smooth_shading=True)
p.add_points(sparse, color="#D95319", point_size=20, render_points_as_spheres=True)
p.camera_position = "xz"; p.camera.zoom(1.3)
p.screenshot(str(OUT_DIR / "mean_sparse_landmarks.png")); p.close()
print("wrote", OUT_DIR / "mean_sparse_landmarks.png")

### Dense correspondences

In [ ]:
# Dense correspondences - html interactive
 
v, f = mesh.points, mesh.faces.reshape(-1, 4)[:, 1:]
fig = go.Figure([
    go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2],
              color="lightgray", opacity=0.45, hoverinfo="skip", showlegend=False),
    go.Scatter3d(x=dense[:,0], y=dense[:,1], z=dense[:,2], mode="markers",
                 marker=dict(size=1.6, color="#0072BD"), name=f"dense (n={len(dense):,})")])
fig.update_layout(title="Mean dense correspondences on atlas mesh", width=900, height=700,
                  scene=dict(aspectmode="data"), margin=dict(l=0, r=0, b=0, t=40))
fig.write_html(str(OUT_DIR / "mean_dense_correspondences.html"), include_plotlyjs="cdn")


In [ ]:
# In notebook 3d render  

p = pv.Plotter(window_size=(900, 700))
p.add_mesh(mesh, color="lightgray", opacity=0.45, smooth_shading=True)
p.add_points(dense,  color="#0072BD", point_size=4,
             render_points_as_spheres=True, label="dense")
p.add_legend()
p.show()

In [ ]:
# Screenshot -> PNG  
p = pv.Plotter(off_screen=True, window_size=(1600, 1200))
p.add_mesh(mesh, color="lightgray", opacity=0.45, smooth_shading=True)
p.add_points(dense,  color="#0072BD", point_size=15,  render_points_as_spheres=True)
p.camera_position = "xz"; p.camera.zoom(1.3)
p.screenshot(str(OUT_DIR / "mean_dense_correspondences.png")); p.close()
print("wrote", OUT_DIR / "mean_dense_correspondences.png")

### Mean mesh only (no landmarks/correspondences)

In [ ]:
# Screenshot - mean mesh no lms

p = pv.Plotter(off_screen=True, window_size=(1600, 1200))
p.add_mesh(mesh, color="lightgray", opacity=0.45, smooth_shading=True)
p.camera_position = "xz"; p.camera.zoom(1.3)
p.screenshot(str(OUT_DIR / "mean_mesh.png")); p.close()
print("wrote", OUT_DIR / "mean_mesh.png")